# 05. 앙상블 & 최종 예측 (2026-05-25)

## 앙상블 전략
각 모델의 **검증 RMSE 역수를 가중치**로 사용한 가중평균

```
최종 예측 = w_sarimax × SARIMAX예측 + w_xgb × XGBoost예측 + w_lgb × LightGBM예측
가중치 = (1/RMSE_i) / Σ(1/RMSE_j)
```

## 최종 출력
- 6개구 × 3가지 모델 예측값 비교표
- 앙상블 최종 예측값 + 불확실성 구간
- 시각화 (막대 그래프, 구별 비교)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

DATA_PROC = Path('../data/processed')
TARGET_DISTRICTS = ['노원구', '은평구', '서대문구', '서초구', '강남구', '송파구']
PREDICT_DATE = '2026-05-25'

# 각 모델 예측 결과 로드
try:
    sarimax_df = pd.read_csv(DATA_PROC / 'sarimax_predictions.csv')
    ml_df      = pd.read_csv(DATA_PROC / 'ml_predictions.csv')
    print('모델 예측 파일 로드 완료')
    print('SARIMAX:', sarimax_df.shape)
    print('ML:', ml_df.shape)
except FileNotFoundError as e:
    print(f'파일 없음: {e}')
    print('노트북 03, 04를 먼저 실행해주세요.')

    # 데모용 더미 예측값 생성 (실제 과제에서는 삭제)
    dummy_data = {
        '구': TARGET_DISTRICTS,
        'SARIMAX 예측': [98.1, 99.0, 101.1, 115.7, 118.8, 113.1],
        '95% CI 하한':  [97.0, 97.8, 99.5, 113.2, 116.0, 110.9],
        '95% CI 상한':  [99.2, 100.2, 102.7, 118.2, 121.6, 115.3],
    }
    sarimax_df = pd.DataFrame(dummy_data)
    ml_df = pd.DataFrame({
        '구': TARGET_DISTRICTS,
        'XGBoost':  [97.9, 98.8, 100.9, 116.1, 119.2, 113.5],
        'LightGBM': [98.0, 99.1, 101.0, 115.9, 118.9, 113.3],
    })

sarimax_df = sarimax_df.set_index('구')
ml_df      = ml_df.set_index('구')

## 1. 모델별 검증 성능 기반 앙상블 가중치 설정

In [ ]:
# 검증 RMSE 입력 (노트북 03, 04 실행 후 실제값으로 업데이트)
# 예시값: 실제 검증 결과로 교체하세요
validation_rmse = {
    'SARIMAX':   0.35,  # 03_arima_sarimax.ipynb의 평균 RMSE
    'XGBoost':   0.28,  # 04_xgboost_lgbm.ipynb의 평균 RMSE
    'LightGBM':  0.30,  # 04_xgboost_lgbm.ipynb의 평균 RMSE
}

# 가중치 = 1/RMSE 정규화
inv_rmse = {k: 1.0 / v for k, v in validation_rmse.items()}
total    = sum(inv_rmse.values())
weights  = {k: v / total for k, v in inv_rmse.items()}

print('앙상블 가중치:')
for model, w in weights.items():
    print(f'  {model}: {w:.3f} (검증 RMSE={validation_rmse[model]})')

## 2. 앙상블 최종 예측

In [ ]:
result_rows = []

for district in TARGET_DISTRICTS:
    if district not in sarimax_df.index or district not in ml_df.index:
        continue

    sarimax_val = float(sarimax_df.loc[district, 'SARIMAX 예측'])
    xgb_val     = float(ml_df.loc[district, 'XGBoost'])
    lgb_val     = float(ml_df.loc[district, 'LightGBM'])

    ensemble_val = (
        weights['SARIMAX']  * sarimax_val +
        weights['XGBoost']  * xgb_val +
        weights['LightGBM'] * lgb_val
    )

    # 불확실성: SARIMAX CI 활용 + ML 모델 간 편차
    ci_low  = float(sarimax_df.loc[district, '95% CI 하한'])
    ci_high = float(sarimax_df.loc[district, '95% CI 상한'])
    model_spread = np.std([sarimax_val, xgb_val, lgb_val])

    result_rows.append({
        '구':         district,
        'SARIMAX':    round(sarimax_val, 2),
        'XGBoost':    round(xgb_val, 2),
        'LightGBM':   round(lgb_val, 2),
        '앙상블 최종': round(ensemble_val, 2),
        '95%CI 하한':  round(min(ci_low, ensemble_val - 2*model_spread), 2),
        '95%CI 상한':  round(max(ci_high, ensemble_val + 2*model_spread), 2),
    })

final_df = pd.DataFrame(result_rows).set_index('구')
final_df.to_csv(DATA_PROC / 'final_predictions.csv')

print(f'\n====== 최종 예측 결과: {PREDICT_DATE} ======')
print(final_df.to_string())

## 3. 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 그래프 1: 모델별 예측값 비교 (군집 막대) ---
ax = axes[0]
x = np.arange(len(TARGET_DISTRICTS))
width = 0.2
models = ['SARIMAX', 'XGBoost', 'LightGBM', '앙상블 최종']
colors_bar = ['steelblue', 'tomato', 'seagreen', 'gold']

for i, (model, color) in enumerate(zip(models, colors_bar)):
    vals = [final_df.loc[d, model] if d in final_df.index else np.nan
            for d in TARGET_DISTRICTS]
    ax.bar(x + i * width, vals, width, label=model, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(TARGET_DISTRICTS, rotation=15)
ax.set_ylabel('아파트 매매가격지수')
ax.set_title(f'모델별 예측값 비교 ({PREDICT_DATE})')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# --- 그래프 2: 앙상블 예측 + 신뢰구간 ---
ax2 = axes[1]
districts = [d for d in TARGET_DISTRICTS if d in final_df.index]
ensemble_vals = [final_df.loc[d, '앙상블 최종'] for d in districts]
ci_lows  = [final_df.loc[d, '95%CI 하한'] for d in districts]
ci_highs = [final_df.loc[d, '95%CI 상한'] for d in districts]

y_pos = np.arange(len(districts))
ax2.barh(y_pos, ensemble_vals, color='gold', alpha=0.8, label='앙상블 예측')
ax2.errorbar(
    ensemble_vals, y_pos,
    xerr=[np.array(ensemble_vals) - np.array(ci_lows),
          np.array(ci_highs) - np.array(ensemble_vals)],
    fmt='none', color='black', capsize=5, linewidth=2
)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(districts)
ax2.set_xlabel('아파트 매매가격지수')
ax2.set_title(f'앙상블 최종 예측 + 95% 신뢰구간 ({PREDICT_DATE})')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_PROC / 'final_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n시각화 저장 완료: data/processed/final_forecast.png')

In [ ]:
# 최종 결과 요약 출력
print('=' * 60)
print(f'서울 6개구 아파트 매매가격지수 예측 ({PREDICT_DATE})')
print('=' * 60)
for district in TARGET_DISTRICTS:
    if district not in final_df.index:
        continue
    row = final_df.loc[district]
    print(f"\n{district}")
    print(f"  SARIMAX  : {row['SARIMAX']:.2f}")
    print(f"  XGBoost  : {row['XGBoost']:.2f}")
    print(f"  LightGBM : {row['LightGBM']:.2f}")
    print(f"  앙상블   : {row['앙상블 최종']:.2f}  [{row['95%CI 하한']:.2f}, {row['95%CI 상한']:.2f}]")
print('\n* CI = 95% 신뢰구간')
print('* 앙상블 가중치: SARIMAX={:.2f}, XGBoost={:.2f}, LightGBM={:.2f}'.format(
    weights['SARIMAX'], weights['XGBoost'], weights['LightGBM']))